# T1 - Tic Tac Toe

## 1. Importação das bibliotecas



In [1]:
import pandas as pd
import numpy as np

## 2. Carregamento do dataset original

In [2]:
columns = [
    'top_left',
    'top_middle',
    'top_right',
    'middle_left',
    'middle_middle',
    'middle_right',
    'bottom_left',
    'bottom_middle',
    'bottom_right',
    'class'
]

df = pd.read_csv(
    '../datasetOriginal/tic-tac-toe.data',
    names=columns
)

df.head()

,top_left,top_middle,top_right,middle_left,middle_middle,middle_right,bottom_left,bottom_middle,bottom_right,class
0,x,x,x,x,o,o,x,o,o,positive
1,x,x,x,x,o,o,o,x,o,positive
2,x,x,x,x,o,o,o,o,x,positive
3,x,x,x,x,o,o,o,b,b,positive
4,x,x,x,x,o,o,b,o,b,positive


## 3. Análise do dataset

In [3]:
print(f'Quantidade de linhas: {df.shape[0]}')
print(f'Quantidade de colunas: {df.shape[1]}')

print('\nDistribuição da classe original:')
print(df['class'].value_counts())

print('\nValores ausentes:')
print(df.isnull().sum())

print('\nQuantidade de registros duplicados:')
print(df.duplicated().sum())

Quantidade de linhas: 958
Quantidade de colunas: 10

Distribuição da classe original:
class
positive    626
negative    332
Name: count, dtype: int64

Valores ausentes:
top_left         0
top_middle       0
top_right        0
middle_left      0
middle_middle    0
middle_right     0
bottom_left      0
bottom_middle    0
bottom_right     0
class            0
dtype: int64

Quantidade de registros duplicados:
0


In [4]:
for column in columns[:-1]:
    print(column, df[column].unique())

top_left <ArrowStringArray>
['x', 'o', 'b']
Length: 3, dtype: str
top_middle <ArrowStringArray>
['x', 'o', 'b']
Length: 3, dtype: str
top_right <ArrowStringArray>
['x', 'o', 'b']
Length: 3, dtype: str
middle_left <ArrowStringArray>
['x', 'o', 'b']
Length: 3, dtype: str
middle_middle <ArrowStringArray>
['o', 'b', 'x']
Length: 3, dtype: str
middle_right <ArrowStringArray>
['o', 'b', 'x']
Length: 3, dtype: str
bottom_left <ArrowStringArray>
['x', 'o', 'b']
Length: 3, dtype: str
bottom_middle <ArrowStringArray>
['o', 'x', 'b']
Length: 3, dtype: str
bottom_right <ArrowStringArray>
['o', 'x', 'b']
Length: 3, dtype: str


## 4. Ajeitar as classes

In [5]:
board_columns = [
    'top_left',
    'top_middle',
    'top_right',
    'middle_left',
    'middle_middle',
    'middle_right',
    'bottom_left',
    'bottom_middle',
    'bottom_right'
]

winning_lines = [
    [0, 1, 2],
    [3, 4, 5],
    [6, 7, 8],
    [0, 3, 6],
    [1, 4, 7],
    [2, 5, 8],
    [0, 4, 8],
    [2, 4, 6]
]

In [6]:
def classify_board(row):
    board = row[board_columns].values

    for line in winning_lines:
        values = [board[i] for i in line]

        if all(value == 'x' for value in values):
            return 'X_VENCEU'

        if all(value == 'o' for value in values):
            return 'O_VENCEU'

    if 'b' not in board:
        return 'EMPATE'

    return 'TEM_JOGO'

In [7]:
df['game_state'] = df.apply(classify_board, axis=1)

df['game_state'].value_counts()

game_state
X_VENCEU    626
O_VENCEU    316
EMPATE       16
Name: count, dtype: int64

## 5. Validação dos estados do jogo

o dataset só tem estados finais entao a gente precisa representar a classe tem jogo. comecamos com o tabuleiro vazio, o X e o O sao os oponentes e assim que interrompe a partida temos vitória ou empate.

In [8]:
def generate_valid_states():
    valid_states = set()
    visited = set()

    def play(board, player):
        if board in visited:
            return

        visited.add(board)

        state = classify_board(
            pd.Series(dict(zip(board_columns, board)))
        )

        if board != ('b',) * 9:
            valid_states.add(board)

        if state != 'TEM_JOGO':
            return

        next_player = 'o' if player == 'x' else 'x'

        for position in range(9):
            if board[position] == 'b':
                new_board = list(board)
                new_board[position] = player

                play(tuple(new_board), next_player)

    play(('b',) * 9, 'x')

    return valid_states

In [9]:
valid_states = generate_valid_states()

df_valid = pd.DataFrame(
    list(valid_states),
    columns=board_columns
)

df_valid['game_state'] = df_valid.apply(
    classify_board,
    axis=1
)

print(f'Quantidade de estados válidos únicos: {len(df_valid)}')

print('\nDistribuição por classe:')
print(df_valid['game_state'].value_counts())

Quantidade de estados válidos únicos: 5477

Distribuição por classe:
game_state
TEM_JOGO    4519
X_VENCEU     626
O_VENCEU     316
EMPATE        16
Name: count, dtype: int64


In [10]:
#verificando se os 958 tabuleiros originais estão realmente contidos nos estados finais que geramos.

original_boards = set(
    map(tuple, df[board_columns].values)
)

generated_final_boards = set(
    map(
        tuple,
        df_valid[df_valid['game_state'] != 'TEM_JOGO'][board_columns].values
    )
)

print('Estados finais originais:', len(original_boards))
print('Estados finais gerados:', len(generated_final_boards))
print('Os conjuntos são iguais:', original_boards == generated_final_boards)

Estados finais originais: 958
Estados finais gerados: 958
Os conjuntos são iguais: True


**Resultado esperado:** os 958 estados finais do dataset original continuam representados, mas agora também temos estados intermediários válidos para a classe `TEM_JOGO`.

## 6. Balanceamento e seleção das amostras

In [11]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_valid,
    test_size=0.20,
    random_state=42,
    stratify=df_valid['game_state']
)

print('Treino:')
print(train_df['game_state'].value_counts())

print('\nTeste:')
print(test_df['game_state'].value_counts())

Treino:
game_state
TEM_JOGO    3614
X_VENCEU     501
O_VENCEU     253
EMPATE        13
Name: count, dtype: int64

Teste:
game_state
TEM_JOGO    905
X_VENCEU    125
O_VENCEU     63
EMPATE        3
Name: count, dtype: int64


In [12]:
train_sampled = pd.concat([
    train_df[train_df['game_state'] == 'TEM_JOGO'].sample(
        n=200,
        random_state=42
    ),
    train_df[train_df['game_state'] == 'X_VENCEU'].sample(
        n=200,
        random_state=42
    ),
    train_df[train_df['game_state'] == 'O_VENCEU'].sample(
        n=200,
        random_state=42
    )
])

In [13]:
empate_train = train_df[
    train_df['game_state'] == 'EMPATE'
]

empate_oversampled = empate_train.sample(
    n=200,
    replace=True,
    random_state=42
)

In [14]:
train_balanced = pd.concat([
    train_sampled,
    empate_oversampled
])

train_balanced = train_balanced.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(train_balanced['game_state'].value_counts())

game_state
EMPATE      200
TEM_JOGO    200
O_VENCEU    200
X_VENCEU    200
Name: count, dtype: int64


## 7. Pré-processamento - Abordagem 1

In [15]:
board_mapping = {
    'x': 1,
    'o': -1,
    'b': 0
}

X_train_board = train_balanced[board_columns].replace(board_mapping)
X_test_board = test_df[board_columns].replace(board_mapping)

y_train = train_balanced['game_state']
y_test = test_df['game_state']

X_train_board.head()

,top_left,top_middle,top_right,middle_left,middle_middle,middle_right,bottom_left,bottom_middle,bottom_right
0,-1,1,-1,1,-1,1,1,-1,1
1,-1,1,-1,-1,1,1,1,-1,1
2,-1,-1,1,0,0,-1,1,0,1
3,-1,-1,-1,1,1,0,1,0,0
4,0,0,0,0,1,1,0,-1,-1


In [16]:
print('Formato treino:', X_train_board.shape)
print('Formato teste:', X_test_board.shape)

print('\nExemplo:')
print(X_train_board.iloc[0])

Formato treino: (800, 9)
Formato teste: (1096, 9)

Exemplo:
top_left         -1
top_middle        1
top_right        -1
middle_left       1
middle_middle    -1
middle_right      1
bottom_left       1
bottom_middle    -1
bottom_right      1
Name: 0, dtype: object


## 8. Pré-processamento - Abordagem 2

In [17]:
def extract_features(row):
    board = row[board_columns].values

    x_count = np.sum(board == 'x')
    o_count = np.sum(board == 'o')
    empty_count = np.sum(board == 'b')
    occupied = x_count + o_count

    lines_with_2_x = 0
    lines_with_2_o = 0

    for line in winning_lines:
        values = [board[i] for i in line]

        if values.count('x') == 2 and values.count('b') == 1:
            lines_with_2_x += 1

        if values.count('o') == 2 and values.count('b') == 1:
            lines_with_2_o += 1


    next_player = 0 if x_count == o_count else 1

    return pd.Series({
        'x_count': x_count,
        'o_count': o_count,
        'occupied': occupied,
        'lines_with_2_x': lines_with_2_x,
        'lines_with_2_o': lines_with_2_o,
        'empty_count': empty_count,
        'next_player': next_player
    })

In [18]:
X_train_features = train_balanced.apply(
    extract_features,
    axis=1
)

X_test_features = test_df.apply(
    extract_features,
    axis=1
)

X_train_features.head()

,x_count,o_count,occupied,lines_with_2_x,lines_with_2_o,empty_count,next_player
0,5,4,9,0,0,0,1
1,5,4,9,0,0,0,1
2,3,3,6,2,0,3,0
3,3,3,6,1,0,3,0
4,2,2,4,1,1,5,0


In [19]:
print('Formato treino:', X_train_features.shape)
print('Formato teste:', X_test_features.shape)

Formato treino: (800, 7)
Formato teste: (1096, 7)


In [20]:
display(X_train_board.head())
display(X_train_features.head())

,top_left,top_middle,top_right,middle_left,middle_middle,middle_right,bottom_left,bottom_middle,bottom_right
0,-1,1,-1,1,-1,1,1,-1,1
1,-1,1,-1,-1,1,1,1,-1,1
2,-1,-1,1,0,0,-1,1,0,1
3,-1,-1,-1,1,1,0,1,0,0
4,0,0,0,0,1,1,0,-1,-1


,x_count,o_count,occupied,lines_with_2_x,lines_with_2_o,empty_count,next_player
0,5,4,9,0,0,0,1
1,5,4,9,0,0,0,1
2,3,3,6,2,0,3,0
3,3,3,6,1,0,3,0
4,2,2,4,1,1,5,0


## 9. k-NN


In [21]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

X_train_board_cv = train_df[board_columns].replace(board_mapping)
X_train_features_cv = train_df.apply(extract_features, axis=1)
y_train_cv = train_df['game_state']

knn_params = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15],
    'knn__weights': ['uniform', 'distance']
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [22]:
#abordagem 1
knn_board = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

grid_knn_board = GridSearchCV(
    knn_board,
    knn_params,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1
)

grid_knn_board.fit(
    X_train_board_cv,
    y_train_cv
)

print('Melhores parâmetros:')
print(grid_knn_board.best_params_)

print('Melhor F1 na validação:')
print(grid_knn_board.best_score_)

best_knn_board = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(
        n_neighbors=grid_knn_board.best_params_['knn__n_neighbors'],
        weights=grid_knn_board.best_params_['knn__weights']
    ))
])

best_knn_board.fit(
    X_train_board,
    y_train
)

pred_knn_board = best_knn_board.predict(X_test_board)

knn_board_accuracy = accuracy_score(y_test, pred_knn_board)
knn_board_precision = precision_score(
    y_test, pred_knn_board, average='macro', zero_division=0
)
knn_board_recall = recall_score(
    y_test, pred_knn_board, average='macro', zero_division=0
)
knn_board_f1 = f1_score(
    y_test, pred_knn_board, average='macro', zero_division=0
)

print('Accuracy:', knn_board_accuracy)
print('Precision:', knn_board_precision)
print('Recall:', knn_board_recall)
print('F1:', knn_board_f1)

print('\nClassification report:')
print(classification_report(y_test, pred_knn_board, zero_division=0))

Melhores parâmetros:
{'knn__n_neighbors': 9, 'knn__weights': 'uniform'}
Melhor F1 na validação:
0.4202113182090164
Accuracy: 0.5291970802919708
Precision: 0.3851770703677812
Recall: 0.5457430500745417
F1: 0.3710646348912143

Classification report:
              precision    recall  f1-score   support

      EMPATE       0.00      0.00      0.00         3
    O_VENCEU       0.23      0.97      0.37        63
    TEM_JOGO       0.96      0.47      0.63       905
    X_VENCEU       0.35      0.74      0.48       125

    accuracy                           0.53      1096
   macro avg       0.39      0.55      0.37      1096
weighted avg       0.84      0.53      0.60      1096



In [23]:
# abordagem 2
knn_features = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

grid_knn_features = GridSearchCV(
    knn_features,
    knn_params,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1
)

grid_knn_features.fit(
    X_train_features_cv,
    y_train_cv
)

print('Melhores parâmetros:')
print(grid_knn_features.best_params_)

print('Melhor F1 na validação:')
print(grid_knn_features.best_score_)
best_knn_features = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(
        n_neighbors=grid_knn_features.best_params_['knn__n_neighbors'],
        weights=grid_knn_features.best_params_['knn__weights']
    ))
])

best_knn_features.fit(
    X_train_features,
    y_train
)

pred_knn_features = best_knn_features.predict(X_test_features)

knn_features_accuracy = accuracy_score(y_test, pred_knn_features)
knn_features_precision = precision_score(
    y_test, pred_knn_features, average='macro', zero_division=0
)
knn_features_recall = recall_score(
    y_test, pred_knn_features, average='macro', zero_division=0
)
knn_features_f1 = f1_score(
    y_test, pred_knn_features, average='macro', zero_division=0
)

print('Accuracy:', knn_features_accuracy)
print('Precision:', knn_features_precision)
print('Recall:', knn_features_recall)
print('F1:', knn_features_f1)

print('\nClassification report:')
print(classification_report(y_test, pred_knn_features, zero_division=0))

Melhores parâmetros:
{'knn__n_neighbors': 3, 'knn__weights': 'uniform'}
Melhor F1 na validação:
0.5209086897367905
Accuracy: 0.7828467153284672
Precision: 0.5229997263914946
Recall: 0.9170165745856353
F1: 0.6039617869521413

Classification report:
              precision    recall  f1-score   support

      EMPATE       0.23      1.00      0.38         3
    O_VENCEU       0.39      1.00      0.57        63
    TEM_JOGO       1.00      0.75      0.86       905
    X_VENCEU       0.47      0.92      0.62       125

    accuracy                           0.78      1096
   macro avg       0.52      0.92      0.60      1096
weighted avg       0.90      0.78      0.81      1096



In [24]:
# comparação entre as abordagens
knn_results = pd.DataFrame([
    {
        'Abordagem': 'Tabuleiro',
        'Accuracy': knn_board_accuracy,
        'Precision': knn_board_precision,
        'Recall': knn_board_recall,
        'F1': knn_board_f1
    },
    {
        'Abordagem': 'Features',
        'Accuracy': knn_features_accuracy,
        'Precision': knn_features_precision,
        'Recall': knn_features_recall,
        'F1': knn_features_f1
    }
])

knn_results

,Abordagem,Accuracy,Precision,Recall,F1
0,Tabuleiro,0.529197,0.385177,0.545743,0.371065
1,Features,0.782847,0.523000,0.917017,0.603962


## 10. MLP

## 11. Árvores de Decisão

## 12. SVM

## 13. Random Forest

In [25]:
from sklearn.ensemble import RandomForestClassifier

rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}


In [26]:
rf_board = RandomForestClassifier(
    random_state=42
)

grid_rf_board = GridSearchCV(
    rf_board,
    rf_params,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1
)

grid_rf_board.fit(
    X_train_board_cv,
    y_train_cv
)

print('Melhores parâmetros:')
print(grid_rf_board.best_params_)

print('Melhor F1 na validação:')
print(grid_rf_board.best_score_)


Melhores parâmetros:
{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Melhor F1 na validação:
0.6139192822087833


In [27]:
best_rf_board = RandomForestClassifier(
    **grid_rf_board.best_params_,
    random_state=42
)

best_rf_board.fit(
    X_train_board,
    y_train
)

pred_rf_board = best_rf_board.predict(X_test_board)

rf_board_accuracy = accuracy_score(y_test, pred_rf_board)
rf_board_precision = precision_score(
    y_test, pred_rf_board, average='macro', zero_division=0
)
rf_board_recall = recall_score(
    y_test, pred_rf_board, average='macro', zero_division=0
)
rf_board_f1 = f1_score(
    y_test, pred_rf_board, average='macro', zero_division=0
)

print('Accuracy:', rf_board_accuracy)
print('Precision:', rf_board_precision)
print('Recall:', rf_board_recall)
print('F1:', rf_board_f1)

print('\nClassification report:')
print(classification_report(y_test, pred_rf_board, zero_division=0))


Accuracy: 0.7299270072992701
Precision: 0.44241205918837495
Recall: 0.6530579672016136
F1: 0.47572605675412316

Classification report:
              precision    recall  f1-score   support

      EMPATE       0.00      0.00      0.00         3
    O_VENCEU       0.28      0.97      0.44        63
    TEM_JOGO       0.99      0.68      0.81       905
    X_VENCEU       0.50      0.96      0.66       125

    accuracy                           0.73      1096
   macro avg       0.44      0.65      0.48      1096
weighted avg       0.89      0.73      0.77      1096



In [28]:
rf_features = RandomForestClassifier(
    random_state=42
)

grid_rf_features = GridSearchCV(
    rf_features,
    rf_params,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1
)

grid_rf_features.fit(
    X_train_features_cv,
    y_train_cv
)

print('Melhores parâmetros:')
print(grid_rf_features.best_params_)

print('Melhor F1 na validação:')
print(grid_rf_features.best_score_)


Melhores parâmetros:
{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Melhor F1 na validação:
0.4101474018418494


In [29]:
best_rf_features = RandomForestClassifier(
    **grid_rf_features.best_params_,
    random_state=42
)

best_rf_features.fit(
    X_train_features,
    y_train
)

pred_rf_features = best_rf_features.predict(X_test_features)

rf_features_accuracy = accuracy_score(y_test, pred_rf_features)
rf_features_precision = precision_score(
    y_test, pred_rf_features, average='macro', zero_division=0
)
rf_features_recall = recall_score(
    y_test, pred_rf_features, average='macro', zero_division=0
)
rf_features_f1 = f1_score(
    y_test, pred_rf_features, average='macro', zero_division=0
)

print('Accuracy:', rf_features_accuracy)
print('Precision:', rf_features_precision)
print('Recall:', rf_features_recall)
print('F1:', rf_features_f1)

print('\nClassification report:')
print(classification_report(y_test, pred_rf_features, zero_division=0))


Accuracy: 0.7846715328467153
Precision: 0.5239576765447667
Recall: 0.9175690607734807
F1: 0.6051629717275566

Classification report:
              precision    recall  f1-score   support

      EMPATE       0.23      1.00      0.38         3
    O_VENCEU       0.39      1.00      0.57        63
    TEM_JOGO       1.00      0.75      0.86       905
    X_VENCEU       0.47      0.92      0.62       125

    accuracy                           0.78      1096
   macro avg       0.52      0.92      0.61      1096
weighted avg       0.90      0.78      0.81      1096



In [30]:
rf_results = pd.DataFrame([
    {
        'Abordagem': 'Tabuleiro',
        'Accuracy': rf_board_accuracy,
        'Precision': rf_board_precision,
        'Recall': rf_board_recall,
        'F1': rf_board_f1
    },
    {
        'Abordagem': 'Features',
        'Accuracy': rf_features_accuracy,
        'Precision': rf_features_precision,
        'Recall': rf_features_recall,
        'F1': rf_features_f1
    }
])

rf_results


,Abordagem,Accuracy,Precision,Recall,F1
0,Tabuleiro,0.729927,0.442412,0.653058,0.475726
1,Features,0.784672,0.523958,0.917569,0.605163


In [31]:
rf_weighted_params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2, 4]
}

rf_weighted = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
)

grid_rf_weighted = GridSearchCV(
    rf_weighted,
    rf_weighted_params,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1
)

grid_rf_weighted.fit(
    X_train_features_cv,
    y_train_cv
)

print('Melhores parâmetros:')
print(grid_rf_weighted.best_params_)
print('Melhor F1 na validação:')
print(grid_rf_weighted.best_score_)


Melhores parâmetros:
{'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Melhor F1 na validação:
0.6045964065052328


In [32]:
best_rf_weighted = grid_rf_weighted.best_estimator_

pred_rf_weighted = best_rf_weighted.predict(X_test_features)

rf_weighted_accuracy = accuracy_score(y_test, pred_rf_weighted)
rf_weighted_precision = precision_score(
    y_test, pred_rf_weighted, average='macro', zero_division=0
)
rf_weighted_recall = recall_score(
    y_test, pred_rf_weighted, average='macro', zero_division=0
)
rf_weighted_f1 = f1_score(
    y_test, pred_rf_weighted, average='macro', zero_division=0
)

print('Accuracy:', rf_weighted_accuracy)
print('Precision:', rf_weighted_precision)
print('Recall:', rf_weighted_recall)
print('F1:', rf_weighted_f1)
print('\nClassification report:')
print(classification_report(y_test, pred_rf_weighted, zero_division=0))


Accuracy: 0.7974452554744526
Precision: 0.5322898304831858
Recall: 0.9214364640883977
F1: 0.6157468365766852

Classification report:
              precision    recall  f1-score   support

      EMPATE       0.23      1.00      0.38         3
    O_VENCEU       0.42      1.00      0.59        63
    TEM_JOGO       1.00      0.77      0.87       905
    X_VENCEU       0.48      0.92      0.63       125

    accuracy                           0.80      1096
   macro avg       0.53      0.92      0.62      1096
weighted avg       0.91      0.80      0.82      1096



In [33]:
rf_final_results = pd.DataFrame([
    {
        'Modelo': 'Random Forest balanceado por amostragem',
        'Accuracy': rf_features_accuracy,
        'Precision': rf_features_precision,
        'Recall': rf_features_recall,
        'F1': rf_features_f1
    },
    {
        'Modelo': 'Random Forest class_weight',
        'Accuracy': rf_weighted_accuracy,
        'Precision': rf_weighted_precision,
        'Recall': rf_weighted_recall,
        'F1': rf_weighted_f1
    }
])

rf_final_results


,Modelo,Accuracy,Precision,Recall,F1
0,Random Forest balanceado por amostragem,0.784672,0.523958,0.917569,0.605163
1,Random Forest class_weight,0.797445,0.532290,0.921436,0.615747


## 14. Comparação dos resultados

## 15. Exportação dos resultados

In [34]:
import joblib
from pathlib import Path

pasta_modelos = Path('/workspaces/T1-IA/modelos')
pasta_modelos.mkdir(parents=True, exist_ok=True)

modelos = {
    'knn': 'best_knn_board',
    'random_forest': 'best_rf_board',
    'svm': 'best_svm_board',
    'mlp': 'best_mlp_board',
    'arvore_decisao': 'best_dt_board'
}

for nome, variavel in modelos.items():
    if variavel in globals():
        caminho = pasta_modelos / f'{nome}.pkl'

        if caminho.exists():
            caminho.unlink()

        joblib.dump(globals()[variavel], caminho)

        print(f'✅ {nome} exportado: {caminho}')
    else:
        print(f'⏭ {nome} ainda não disponível')

✅ knn exportado: /workspaces/T1-IA/modelos/knn.pkl
✅ random_forest exportado: /workspaces/T1-IA/modelos/random_forest.pkl
⏭ svm ainda não disponível
⏭ mlp ainda não disponível
⏭ arvore_decisao ainda não disponível
